# Exhibit 1 Replication — Hedge Fund Returns

Pull in the appicable data to replicate the table from exhibit 1 in the article 'Do Hedge Funds Hedge'

In [3]:
import pandas as pd
import numpy as np

# Read the excess-return data from your workbook
df = pd.read_excel(
    "Exhibit 1 Replication Workbook.xlsx",
    sheet_name="NAVROR",
    usecols="O:Y",
    skiprows=3,
    header=None
)

# Aggregate hedge fund index excess returns
returns = df.iloc[:, 0]

# Calculate performance metrics
annual_return = returns.mean() * 12
annual_volatility = returns.std() * np.sqrt(12)
sharpe_ratio = annual_return / annual_volatility

print("Annual Return:", annual_return)
print("Annual Volatility:", annual_volatility)
print("Sharpe Ratio:", sharpe_ratio)

Annual Return: 0.07972305523083535
Annual Volatility: 0.09994735204996417
Sharpe Ratio: 0.7976504989444984


In [14]:
import pandas as pd
import numpy as np

file = "Exhibit 1 Replication Workbook.xlsx"

raw = pd.read_excel(
    file,
    sheet_name="NAVROR",
    header=None,
    skiprows=3,
    usecols="A:M",
    nrows=81
)





In [11]:
raw.columns = [
    "Date",
    "Agg",
    "Convert",
    "EventDr",
    "EqNtrl",
    "FI",
    "LongShrt",
    "EM",
    "GlblMcro",
    "ManFut",
    "DedShrt",
    "SnP",
    "RiskFree"
]

display(raw.head())


,Date,Agg,Convert,EventDr,EqNtrl,FI,LongShrt,EM,GlblMcro,ManFut,DedShrt,SnP,RiskFree
0,2000-09-30,-0.0041,0.0202,0.0061,-0.0014,0.0075,-0.0071,-0.0679,-0.0087,-0.0334,0.0989,-0.052960,0.005019
1,2000-08-31,0.0339,0.0169,0.0171,0.0143,0.0067,0.0490,0.0432,0.0292,0.0123,-0.0723,0.061948,0.005080
2,2000-07-31,0.0011,0.0146,0.0050,0.0123,0.0055,-0.0063,0.0058,0.0181,-0.0205,0.0359,-0.015801,0.004937
3,2000-06-30,0.0366,0.0197,0.0194,0.0182,0.0093,0.0613,0.0553,0.0132,-0.0112,-0.0699,0.024487,0.004780
4,2000-05-31,-0.0117,0.0194,0.0002,0.0146,0.0002,-0.0345,-0.0413,0.0232,0.0058,0.0614,-0.020684,0.004933


In [13]:

# Select all return columns, excluding Date and RiskFree
returns = raw.drop(columns=["Date", "RiskFree"])

# Subtract each month's risk-free rate
excess_returns = returns.sub(raw["RiskFree"], axis=0)

# Add the dates back
working_data = pd.concat(
    [raw[["Date"]], excess_returns],
    axis=1
)

display(working_data.head())

,Date,Agg,Convert,EventDr,EqNtrl,FI,LongShrt,EM,GlblMcro,ManFut,DedShrt,SnP
0,2000-09-30,-0.009119,0.015181,0.001081,-0.006419,0.002481,-0.012119,-0.072919,-0.013719,-0.038419,0.093881,-0.057979
1,2000-08-31,0.028820,0.011820,0.012020,0.009220,0.001620,0.043920,0.038120,0.024120,0.007220,-0.077380,0.056867
2,2000-07-31,-0.003837,0.009663,0.000063,0.007363,0.000563,-0.011237,0.000863,0.013163,-0.025437,0.030963,-0.020738
3,2000-06-30,0.031820,0.014920,0.014620,0.013420,0.004520,0.056520,0.050520,0.008420,-0.015980,-0.074680,0.019707
4,2000-05-31,-0.016633,0.014467,-0.004733,0.009667,-0.004733,-0.039433,-0.046233,0.018267,0.000867,0.056467,-0.025617


In [20]:
# Create a copy so we don't change the original calculations
formatted_data = working_data.copy()

# Format the return columns as percentages
for col in formatted_data.columns:
    if col != "Date":
        formatted_data[col] = formatted_data[col].map(
            lambda x: f"{x:.2%}"
        )

display(formatted_data.head())

,Date,Agg,Convert,EventDr,EqNtrl,FI,LongShrt,EM,GlblMcro,ManFut,DedShrt,SnP
0,2000-09-30,-0.91%,1.52%,0.11%,-0.64%,0.25%,-1.21%,-7.29%,-1.37%,-3.84%,9.39%,-5.80%
1,2000-08-31,2.88%,1.18%,1.20%,0.92%,0.16%,4.39%,3.81%,2.41%,0.72%,-7.74%,5.69%
2,2000-07-31,-0.38%,0.97%,0.01%,0.74%,0.06%,-1.12%,0.09%,1.32%,-2.54%,3.10%,-2.07%
3,2000-06-30,3.18%,1.49%,1.46%,1.34%,0.45%,5.65%,5.05%,0.84%,-1.60%,-7.47%,1.97%
4,2000-05-31,-1.66%,1.45%,-0.47%,0.97%,-0.47%,-3.94%,-4.62%,1.83%,0.09%,5.65%,-2.56%


In [22]:
# Select the monthly excess returns
returns = working_data.drop(columns=["Date"])

# 1. Annualized excess returns
annual_returns = returns.mean() * 12

# 2. Annualized standard deviations
annual_volatility = returns.std(ddof=1) * np.sqrt(12)

# 3. Sharpe ratios
sharpe_ratios = annual_returns / annual_volatility

# 4. Correlation with S&P 500
correlations = returns.corrwith(returns["SnP"])

In [23]:
performance = pd.DataFrame({
    "Ann Ex Rtrn": annual_returns,
    "Ann Std Dev": annual_volatility,
    "Sharpe Ratio": sharpe_ratios,
    "Correl SnP": correlations
}).T

display(performance.round(4))

,Agg,Convert,EventDr,EqNtrl,FI,LongShrt,EM,GlblMcro,ManFut,DedShrt,SnP
Ann Ex Rtrn,0.0797,0.0541,0.0701,0.0638,0.0158,0.1178,0.0231,0.0772,-0.0116,-0.0712,0.1455
Ann Std Dev,0.0999,0.0507,0.0669,0.0346,0.0440,0.1261,0.2080,0.1444,0.1109,0.1856,0.1419
Sharpe Ratio,0.7977,1.0666,1.0479,1.8429,0.3586,0.9345,0.1109,0.5350,-0.1048,-0.3834,1.0257
Correl SnP,0.5241,0.1263,0.5972,0.4816,0.0813,0.6183,0.5018,0.3609,0.0127,-0.7593,1.0000


In [25]:
# Maximum monthly excess return for each index
max_returns = returns.max()

# Minimum monthly excess return for each index
min_returns = returns.min()

# Find the row where each maximum and minimum occurred
max_indices = returns.idxmax()
min_indices = returns.idxmin()

In [26]:
# Find dates corresponding to maximum returns
max_dates = pd.Series({
    col: working_data.loc[max_indices[col], "Date"]
    for col in returns.columns
})

# Find dates corresponding to minimum returns
min_dates = pd.Series({
    col: working_data.loc[min_indices[col], "Date"]
    for col in returns.columns
})

In [27]:
performance.loc["Max Monthly Return"] = max_returns
performance.loc["Min Monthly Return"] = min_returns

# Dates are stored separately because they aren't numeric returns
display(performance.round(4))

print("Dates of maximum returns:")
display(max_dates)

print("Dates of minimum returns:")
display(min_dates)

,Agg,Convert,EventDr,EqNtrl,FI,LongShrt,EM,GlblMcro,ManFut,DedShrt,SnP
Ann Ex Rtrn,0.0797,0.0541,0.0701,0.0638,0.0158,0.1178,0.0231,0.0772,-0.0116,-0.0712,0.1455
Ann Std Dev,0.0999,0.0507,0.0669,0.0346,0.0440,0.1261,0.2080,0.1444,0.1109,0.1856,0.1419
Sharpe Ratio,0.7977,1.0666,1.0479,1.8429,0.3586,0.9345,0.1109,0.5350,-0.1048,-0.3834,1.0257
Correl SnP,0.5241,0.1263,0.5972,0.4816,0.0813,0.6183,0.5018,0.3609,0.0127,-0.7593,1.0000
Max Monthly Return,0.0809,0.0310,0.0343,0.0284,0.0155,0.1257,0.1605,0.1015,0.0954,0.2230,0.0929
Min Monthly Return,-0.0796,-0.0505,-0.1218,-0.0158,-0.0730,-0.1184,-0.2344,-0.1189,-0.0979,-0.0911,-0.1489


Dates of maximum returns:


Agg        1999-12-31
Convert    2000-04-30
EventDr    1994-01-31
EqNtrl     1997-07-31
FI         1995-04-30
LongShrt   1999-12-31
EM         1994-08-31
GlblMcro   1995-08-31
ManFut     1998-08-31
DedShrt    1998-08-31
SnP        2000-03-31
dtype: datetime64[us]

Dates of minimum returns:


Agg        1998-08-31
Convert    1998-08-31
EventDr    1998-08-31
EqNtrl     1997-03-31
FI         1998-10-31
LongShrt   1998-08-31
EM         1998-08-31
GlblMcro   1998-10-31
ManFut     1995-09-30
DedShrt    2000-02-29
SnP        1998-08-31
dtype: datetime64[us]

In [28]:
# Calculate correlation between all return series
correlation_matrix = returns.corr()

# Display the matrix
display(correlation_matrix.round(4))

,Agg,Convert,EventDr,EqNtrl,FI,LongShrt,EM,GlblMcro,ManFut,DedShrt,SnP
Agg,1.0000,0.4066,0.6604,0.3303,0.4815,0.7693,0.6525,0.9008,0.1379,-0.4819,0.5241
Convert,0.4066,1.0000,0.5906,0.3256,0.6428,0.2680,0.3811,0.3131,-0.3425,-0.2543,0.1263
EventDr,0.6604,0.5906,1.0000,0.4247,0.4389,0.6677,0.7037,0.4153,-0.2052,-0.6515,0.5972
EqNtrl,0.3303,0.3256,0.4247,1.0000,0.0427,0.3570,0.2471,0.2173,0.2172,-0.4300,0.4816
FI,0.4815,0.6428,0.4389,0.0427,1.0000,0.2337,0.3399,0.4700,-0.2346,-0.0913,0.0813
LongShrt,0.7693,0.2680,0.6677,0.3570,0.2337,1.0000,0.5931,0.4682,-0.0373,-0.7696,0.6183
EM,0.6525,0.3811,0.7037,0.2471,0.3399,0.5931,1.0000,0.4292,-0.1058,-0.5639,0.5018
GlblMcro,0.9008,0.3131,0.4153,0.2173,0.4700,0.4682,0.4292,1.0000,0.2361,-0.1774,0.3609
ManFut,0.1379,-0.3425,-0.2052,0.2172,-0.2346,-0.0373,-0.1058,0.2361,1.0000,0.1872,0.0127
DedShrt,-0.4819,-0.2543,-0.6515,-0.4300,-0.0913,-0.7696,-0.5639,-0.1774,0.1872,1.0000,-0.7593


# My Output Table

In [39]:
# Match the portfolio names used in the original exhibit
portfolio_names = {
    "Agg": "Aggregate Hedge Fund Index",
    "Convert": "Convertible Arbitrage",
    "EventDr": "Event-Driven",
    "EqNtrl": "Equity Market-Neutral",
    "FI": "Fixed-Income Arbitrage",
    "LongShrt": "Long/Short Equity",
    "EM": "Emerging Markets",
    "GlblMcro": "Global Macro",
    "ManFut": "Managed Futures",
    "DedShrt": "Dedicated Short Bias",
    "SnP": "S&P 500"
}

# Assemble all our calculated statistics
exhibit_replication = pd.DataFrame({
    "Annualized Excess Return": annual_returns,
    "Annualized Standard Deviation": annual_volatility,
    "Sharpe Ratio": sharpe_ratios,
    "Correlation with S&P 500": correlations,
    "Maximum Monthly Return": max_returns,
    "Month of Maximum Return": max_dates,
    "Minimum Monthly Return": min_returns,
    "Month of Minimum Return": min_dates
})

# Replace abbreviations with portfolio names
exhibit_replication.index = exhibit_replication.index.map(
    portfolio_names
)

exhibit_replication.index.name = "Portfolio"

with pd.option_context("display.float_format", "{:.2f}".format):
    display(exhibit_replication)

,Annualized Excess Return,Annualized Standard Deviation,Sharpe Ratio,Correlation with S&P 500,Maximum Monthly Return,Month of Maximum Return,Minimum Monthly Return,Month of Minimum Return
Portfolio,,,,,,,,
Aggregate Hedge Fund Index,0.08,0.10,0.80,0.52,0.08,1999-12-31,-0.08,1998-08-31
Convertible Arbitrage,0.05,0.05,1.07,0.13,0.03,2000-04-30,-0.05,1998-08-31
Event-Driven,0.07,0.07,1.05,0.60,0.03,1994-01-31,-0.12,1998-08-31
Equity Market-Neutral,0.06,0.03,1.84,0.48,0.03,1997-07-31,-0.02,1997-03-31
Fixed-Income Arbitrage,0.02,0.04,0.36,0.08,0.02,1995-04-30,-0.07,1998-10-31
Long/Short Equity,0.12,0.13,0.93,0.62,0.13,1999-12-31,-0.12,1998-08-31
Emerging Markets,0.02,0.21,0.11,0.50,0.16,1994-08-31,-0.23,1998-08-31
Global Macro,0.08,0.14,0.54,0.36,0.10,1995-08-31,-0.12,1998-10-31
Managed Futures,-0.01,0.11,-0.10,0.01,0.10,1998-08-31,-0.10,1995-09-30


# Authors Results

In [38]:
# Read the published results
authors = pd.read_excel(
    file,
    sheet_name="Exhibit 1",
    header=None
)

# Extract the Authors section (Excel rows 6–16)
authors = authors.iloc[5:16, 0:9].copy()

# Match our Python column names
authors.columns = ["Portfolio"] + list(
    exhibit_replication.columns
)

authors = authors.set_index("Portfolio")

with pd.option_context("display.float_format", "{:.2f}".format):
    display(authors)

,Annualized Excess Return,Annualized Standard Deviation,Sharpe Ratio,Correlation with S&P 500,Maximum Monthly Return,Month of Maximum Return,Minimum Monthly Return,Month of Minimum Return
Portfolio,,,,,,,,
Aggregate Hedge Fund Index,0.08,0.10,0.80,0.52,0.08,1999-12-31 00:00:00,-0.08,1998-08-31 00:00:00
Convertible Arbitrage,0.05,0.05,1.07,0.13,0.03,2000-04-30 00:00:00,-0.05,1998-08-31 00:00:00
Event-Driven,0.07,0.07,1.05,0.60,0.03,1994-01-31 00:00:00,-0.12,1998-08-31 00:00:00
Equity Market-Neutral,0.06,0.04,1.85,0.48,0.03,1997-07-31 00:00:00,-0.02,1997-03-31 00:00:00
Fixed-Income Arbitrage,0.02,0.04,0.36,0.08,0.01,1995-04-30 00:00:00,-0.07,1998-10-31 00:00:00
Long/Short Equity,0.12,0.13,0.94,0.62,0.13,1999-12-31 00:00:00,-0.12,1998-08-31 00:00:00
Emerging Markets,0.02,0.21,0.11,0.50,0.16,1994-08-31 00:00:00,-0.23,1998-08-31 00:00:00
Global Macro,0.08,0.14,0.54,0.36,0.10,1995-08-31 00:00:00,-0.12,1998-10-31 00:00:00
Managed Futures,-0.01,0.11,-0.10,0.01,0.10,1998-08-31 00:00:00,-0.10,1995-09-30 00:00:00


In [37]:
# Separate the date columns from the numerical columns
date_columns = [
    "Month of Maximum Return",
    "Month of Minimum Return"
]

numeric_columns = [
    col for col in exhibit_replication.columns
    if col not in date_columns
]

# Calculate numerical differences
difference = (
    exhibit_replication[numeric_columns]
    - authors[numeric_columns]
)

# Compare the dates
def normalize_date(value):
    if isinstance(value, (int, float, np.integer, np.floating)):
        return pd.to_datetime(
            value, unit="D", origin="1899-12-30"
        )
    return pd.to_datetime(value)

for col in date_columns:
    difference[col] = [
        "same" if normalize_date(exhibit_replication.loc[p, col])
        == normalize_date(authors.loc[p, col])
        else "different"
        for p in exhibit_replication.index
    ]

# Restore original column order
difference = difference[exhibit_replication.columns]

with pd.option_context("display.float_format", "{:.2f}".format):
    display(difference)

,Annualized Excess Return,Annualized Standard Deviation,Sharpe Ratio,Correlation with S&P 500,Maximum Monthly Return,Month of Maximum Return,Minimum Monthly Return,Month of Minimum Return
Portfolio,,,,,,,,
Aggregate Hedge Fund Index,-0.00,-0.00,-0.00,0.00,-0.00,same,0.00,same
Convertible Arbitrage,0.00,-0.00,-0.00,-0.00,-0.00,same,0.00,same
Event-Driven,0.00,-0.00,-0.00,-0.00,0.00,same,0.00,same
Equity Market-Neutral,-0.00,-0.00,-0.01,0.00,0.00,same,0.00,same
Fixed-Income Arbitrage,-0.00,-0.00,-0.00,0.00,0.00,same,-0.00,same
Long/Short Equity,-0.00,0.00,-0.01,-0.00,-0.00,same,0.00,same
Emerging Markets,0.00,-0.00,0.00,0.00,-0.00,same,-0.00,same
Global Macro,0.00,0.00,-0.00,0.00,0.00,same,0.00,same
Managed Futures,0.00,-0.00,-0.00,0.00,0.00,same,0.00,same


In [41]:
# Combine all three sections
final_exhibit = pd.concat(
    {
        "AUTHORS": authors,
        "REPLICATION": exhibit_replication,
        "DIFFERENCE TO AUTHORS": difference
    },
    names=["Section", "Portfolio"]
)

# Display with exactly 4 decimal places
with pd.option_context("display.float_format", "{:.2f}".format):
    display(final_exhibit)

Annualized Excess Return  \
Section               Portfolio                                             
AUTHORS               Aggregate Hedge Fund Index                     0.08   
                      Convertible Arbitrage                          0.05   
                      Event-Driven                                   0.07   
                      Equity Market-Neutral                          0.06   
                      Fixed-Income Arbitrage                         0.02   
                      Long/Short Equity                              0.12   
                      Emerging Markets                               0.02   
                      Global Macro                                   0.08   
                      Managed Futures                               -0.01   
                      Dedicated Short Bias                          -0.07   
                      S&P 500                                        0.15   
REPLICATION           Aggregate Hedge Fund Index                     0.08   
                      Convertible Arbitrage                          0.05   
                      Event-Driven                                   0.07   
                      Equity Market-Neutral                          0.06   
                      Fixed-Income Arbitrage                         0.02   
                      Long/Short Equity                              0.12   
                      Emerging Markets                               0.02   
                      Global Macro                                   0.08   
                      Managed Futures                               -0.01   
                      Dedicated Short Bias                          -0.07   
                      S&P 500                                        0.15   
DIFFERENCE TO AUTHORS Aggregate Hedge Fund Index                    -0.00   
                      Convertible Arbitrage                          0.00   
                      Event-Driven                                   0.00   
                      Equity Market-Neutral                         -0.00   
                      Fixed-Income Arbitrage                        -0.00   
                      Long/Short Equity                             -0.00   
                      Emerging Markets                               0.00   
                      Global Macro                                   0.00   
                      Managed Futures                                0.00   
                      Dedicated Short Bias                          -0.00   
                      S&P 500                                       -0.00   

                                                 Annualized Standard Deviation  \
Section               Portfolio                                                  
AUTHORS               Aggregate Hedge Fund Index                          0.10   
                      Convertible Arbitrage                               0.05   
                      Event-Driven                                        0.07   
                      Equity Market-Neutral                               0.04   
                      Fixed-Income Arbitrage                              0.04   
                      Long/Short Equity                                   0.13   
                      Emerging Markets                                    0.21   
                      Global Macro                                        0.14   
                      Managed Futures                                     0.11   
                      Dedicated Short Bias                                0.19   
                      S&P 500                                             0.14   
REPLICATION           Aggregate Hedge Fund Index                          0.10   
                      Convertible Arbitrage                               0.05   
                      Event-Driven                                        0.07   
                      Equity Market-Neutr

In [43]:
%pip install jinja2

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:

from IPython.display import display, HTML
import pandas as pd

# Financial statistics that should display with 2 decimal places
numeric_columns = [
    "Annualized Excess Return",
    "Annualized Standard Deviation",
    "Sharpe Ratio",
    "Correlation with S&P 500",
    "Maximum Monthly Return",
    "Minimum Monthly Return"
]

# Create a reusable styling function
def style_table(df, title, background_color):

    # Work on a copy to preserve original calculations
    formatted_df = df.copy()

    # Convert financial statistics to numeric values
    for col in numeric_columns:
        if col in formatted_df.columns:
            formatted_df[col] = pd.to_numeric(
                formatted_df[col],
                errors="coerce"
            )

    # Format numeric columns to 2 decimal places
    formats = {
        col: "{:.2f}"
        for col in numeric_columns
        if col in formatted_df.columns
    }

    return (
        formatted_df.style
        .format(formats, na_rep="—")
        .set_caption(title)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("font-size", "20px"),
                    ("font-weight", "bold"),
                    ("color", "#17324D"),
                    ("padding", "15px"),
                    ("text-align", "left")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#17324D"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("padding", "10px"),
                    ("text-align", "center")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("background-color", background_color),
                    ("color", "black"),
                    ("font-weight", "bold"),
                    ("padding", "8px"),
                    ("text-align", "right"),
                    ("border-bottom", "1px solid #E5E7EB")
                ]
            }
        ])
    )


# TABLE 1 — Authors
display(
    style_table(
        authors,
        "EXHIBIT 1A | AUTHORS' RESULTS",
        "#EAF2F8"
    )
)

# Add spacing
display(HTML("<div style='height:45px'></div>"))


# TABLE 2 — Python Replication
display(
    style_table(
        exhibit_replication,
        "EXHIBIT 1B | PYTHON REPLICATION",
        "#EDF7F0"
    )
)

# Add spacing
display(HTML("<div style='height:45px'></div>"))


# TABLE 3 — Differences
display(
    style_table(
        difference,
        "EXHIBIT 1C | DIFFERENCE TO AUTHORS",
        "#F5F5F5"
    )
)


,Annualized Excess Return,Annualized Standard Deviation,Sharpe Ratio,Correlation with S&P 500,Maximum Monthly Return,Month of Maximum Return,Minimum Monthly Return,Month of Minimum Return
Portfolio,,,,,,,,
Aggregate Hedge Fund Index,0.08,0.10,0.80,0.52,0.08,1999-12-31 00:00:00,-0.08,1998-08-31 00:00:00
Convertible Arbitrage,0.05,0.05,1.07,0.13,0.03,2000-04-30 00:00:00,-0.05,1998-08-31 00:00:00
Event-Driven,0.07,0.07,1.05,0.60,0.03,1994-01-31 00:00:00,-0.12,1998-08-31 00:00:00
Equity Market-Neutral,0.06,0.04,1.85,0.48,0.03,1997-07-31 00:00:00,-0.02,1997-03-31 00:00:00
Fixed-Income Arbitrage,0.02,0.04,0.36,0.08,0.01,1995-04-30 00:00:00,-0.07,1998-10-31 00:00:00
Long/Short Equity,0.12,0.13,0.94,0.62,0.13,1999-12-31 00:00:00,-0.12,1998-08-31 00:00:00
Emerging Markets,0.02,0.21,0.11,0.50,0.16,1994-08-31 00:00:00,-0.23,1998-08-31 00:00:00
Global Macro,0.08,0.14,0.54,0.36,0.10,1995-08-31 00:00:00,-0.12,1998-10-31 00:00:00
Managed Futures,-0.01,0.11,-0.10,0.01,0.10,1998-08-31 00:00:00,-0.10,1995-09-30 00:00:00


,Annualized Excess Return,Annualized Standard Deviation,Sharpe Ratio,Correlation with S&P 500,Maximum Monthly Return,Month of Maximum Return,Minimum Monthly Return,Month of Minimum Return
Portfolio,,,,,,,,
Aggregate Hedge Fund Index,0.08,0.10,0.80,0.52,0.08,1999-12-31 00:00:00,-0.08,1998-08-31 00:00:00
Convertible Arbitrage,0.05,0.05,1.07,0.13,0.03,2000-04-30 00:00:00,-0.05,1998-08-31 00:00:00
Event-Driven,0.07,0.07,1.05,0.60,0.03,1994-01-31 00:00:00,-0.12,1998-08-31 00:00:00
Equity Market-Neutral,0.06,0.03,1.84,0.48,0.03,1997-07-31 00:00:00,-0.02,1997-03-31 00:00:00
Fixed-Income Arbitrage,0.02,0.04,0.36,0.08,0.02,1995-04-30 00:00:00,-0.07,1998-10-31 00:00:00
Long/Short Equity,0.12,0.13,0.93,0.62,0.13,1999-12-31 00:00:00,-0.12,1998-08-31 00:00:00
Emerging Markets,0.02,0.21,0.11,0.50,0.16,1994-08-31 00:00:00,-0.23,1998-08-31 00:00:00
Global Macro,0.08,0.14,0.54,0.36,0.10,1995-08-31 00:00:00,-0.12,1998-10-31 00:00:00
Managed Futures,-0.01,0.11,-0.10,0.01,0.10,1998-08-31 00:00:00,-0.10,1995-09-30 00:00:00


,Annualized Excess Return,Annualized Standard Deviation,Sharpe Ratio,Correlation with S&P 500,Maximum Monthly Return,Month of Maximum Return,Minimum Monthly Return,Month of Minimum Return
Portfolio,,,,,,,,
Aggregate Hedge Fund Index,-0.00,-0.00,-0.00,0.00,-0.00,same,0.00,same
Convertible Arbitrage,0.00,-0.00,-0.00,-0.00,-0.00,same,0.00,same
Event-Driven,0.00,-0.00,-0.00,-0.00,0.00,same,0.00,same
Equity Market-Neutral,-0.00,-0.00,-0.01,0.00,0.00,same,0.00,same
Fixed-Income Arbitrage,-0.00,-0.00,-0.00,0.00,0.00,same,-0.00,same
Long/Short Equity,-0.00,0.00,-0.01,-0.00,-0.00,same,0.00,same
Emerging Markets,0.00,-0.00,0.00,0.00,-0.00,same,-0.00,same
Global Macro,0.00,0.00,-0.00,0.00,0.00,same,0.00,same
Managed Futures,0.00,-0.00,-0.00,0.00,0.00,same,0.00,same


In [50]:
%pip install xlsxwriter

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [54]:
%pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------- ----- 1.6/1.8 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 8.3 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
